In [19]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
from typing import Dict, List, Union

# some type hints in the header of the function
def my_plotter(x: List[float], y: Union[List[float], List[List[float]]], layout: Dict = {}, names: List[str] = None) -> None:

    y = [y] if all(isinstance(item, (float, int, np.number)) for item in y) else y

    fig_size = layout.get('figsize', (10, 5))
    plt.figure(figsize=fig_size)

    plot_kwargs = {
        'linestyle': layout.get('linestyle', 'solid'),
        'linewidth': layout.get('linewidth', 2),
        'alpha': layout.get('alpha', 0.8)
    }

    show_legend = names is not None
    if names is not None and len(names) != len(y):
        raise ValueError("Length of names is not matching with number of plotted y lists.")

    for i, y_item in enumerate(y):
        if show_legend:
            plot_kwargs['label'] = names[i]
        if 'colors' in layout and i < len(layout['colors']):
            plot_kwargs['color'] = layout['colors'][i]

        plt.plot(x, y_item, **plot_kwargs)

    # Feliratok és dekorációk testreszabása
    if show_legend:
        plt.legend(fontsize=layout.get('legend_size', 12))

    if 'title' in layout:
        plt.title(layout['title'], fontsize=layout.get('title_size', 18))
        rcParams['axes.titlepad'] = layout.get('title_pad', 20)

    if 'x_label' in layout:
        plt.xlabel(layout['x_label'], fontsize=layout.get('label_size', 14))

    if 'y_label' in layout: # Új feature: y tengely feliratozása
        plt.ylabel(layout['y_label'], fontsize=layout.get('label_size', 14))

    if layout.get('grid', True): # Rács bekapcsolása (alapból True)
        plt.grid(True, linestyle=':', alpha=0.6)

    if 'ylim' in layout: # Tengely korlátok állítása
        plt.ylim(layout['ylim'])

    ax = plt.gca()
    if layout.get('show_zero_line', True):
        ax.axhline(0, linestyle='--', color='black', linewidth=1, alpha=0.5)

    plt.tight_layout()
    plt.show()

In [20]:
"""The pricing function of European call option"""
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> dict:
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    """
    # check conditions
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K_vec ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    delta = N_d1_vec #mennyit változik az opció ár ha a részvényár megváltozik
    gamma = (norm.pdf(d1_vec)/(S0 * sigma * T**0.5)) #delta deriváltja
    vega = S0 * (T**0.5) * norm.pdf(d1_vec) #ár volatilitás szerinti parciális deriváltja
    rho = K_vec * T * np.exp(-r*T) * N_d2_vec #kamatláb szerinti

    call_price = N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec

    return {
            "price": call_price,
            "delta": delta,
            "gamma": gamma,
            "vega": vega,
            "rho": rho
        }



In [21]:
r = 0.036
T = 1
S0 = 248.55
sigma = 0.2154
K = 240

apple_call = black_scholes_eur_call(r, T, S0, sigma, K)
print(apple_call)

#30,21-t mond 39,34 az ára jelenleg


{'price': 30.208455870824764, 'delta': 0.669068704210835, 'gamma': 0.0067720020261235764, 'vega': 90.11359447439867, 'rho': 136.08857056077827}


In [22]:
"""The pricing function of European put option"""

def black_scholes_eur_put(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> dict:
    # check conditions
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K_vec ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(-d1_vec)
    N_d2_vec = norm.cdf(-d2_vec)

    delta = -N_d1_vec #mennyit változik az opció ár ha a részvényár megváltozik
    gamma = (norm.pdf(d1_vec)/(S0 * sigma * T**0.5)) #delta deriváltja
    vega = S0 * (T**0.5) * norm.pdf(d1_vec) #ár volatilitás szerinti parciális deriváltja
    rho = - K_vec * T * np.exp(-r*T) * N_d2_vec #kamatláb szerinti

    put_price = K_vec * np.exp((-1.0)*r*T) * N_d2_vec - N_d1_vec * S0

    return {
            "price": put_price,
            "delta": delta,
            "gamma": gamma,
            "vega": vega,
            "rho": rho
        }

In [23]:
r = 0.036
T = 1
S0 = 248.55
sigma = 0.2154
K = 240

apple_put = black_scholes_eur_put(r, T, S0, sigma, K)
print(apple_put)

#13,17-et mond 22,23 az ára jelenleg

{'price': 13.17212630677433, 'delta': -0.330931295789165, 'gamma': 0.0067720020261235764, 'vega': 90.11359447439867, 'rho': -95.4250998751713}


In [24]:
def check_put_call_parity(
    K: Union[float, List[float], np.ndarray],
    r: float,
    T: float,
    S0: float,
    sigma: float
) -> Union[bool, np.ndarray]:
    """
    Put-Call paritást megvizsgálja a függvény
    """

    call = black_scholes_eur_call(r, T, S0, sigma, K)
    put = black_scholes_eur_put(r, T, S0, sigma, K)

    call_price = call['price']
    put_price = put['price']

    lhs = call_price - put_price
    rhs = S0 - (K * np.exp(-r * T))

    return np.isclose(lhs, rhs)



In [25]:
print(check_put_call_parity(K,r,T,S0,sigma))

#a put-call paritás érvényes

True
